# 01_data: the data, step by step (small sample)

**Purpose:** show how raw clicks become training examples: one row per visit that ended without a purchase, the click words in order, and a label (1 = the shopper came back and bought within 7 days without any message).

**Input:** `data/sample/events_sample.parquet`: raw clicks of 1 in 150 shoppers, October and November 2019 (REES46 multi-category store). The full data (1 in 10 shoppers) is built the same way by `scripts/build_training_data.py` and stays in `data/raw/` (not in git).

**Output:** `data/sample/visits_sample.parquet`: `tokens`, `label`, `days_to_buy`, `next_order_value`, `cart_value`, `user_id`, `start_time`, `decide_time`, `split`.

**Findings (20 Nov):** no `remove_from_cart` in the data; visit ids do not reset when people leave (one lasted 16 days), so we define visits ourselves (60 min without a click). Rules were fixed after an independent data audit; see CLAUDE.md.

**Owner:** TBD (Data role)


## Step 1: read the month in pieces, count click types, keep a sample of users
The file is ~5 GB, so we read 5 million rows at a time. We count every click type over the **whole** month
(to see if `remove_from_cart` exists) and keep only users whose id is divisible by 150 (~0.7% of users).
Sampling **by user** keeps all of a user's sessions together, so later we can split train/test by user without leaks.

In [ ]:
import glob
import pandas as pd

files = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
print("\n".join(files))
MONTH_FILE = [f for f in files if "2019-Oct" in f][0]

KEEP_ONE_IN = 150
COLS = ["event_time", "event_type", "product_id", "category_code", "brand", "price", "user_id", "user_session"]

event_counts = pd.Series(dtype="int64")
parts = []
for chunk in pd.read_csv(MONTH_FILE, usecols=COLS, chunksize=5_000_000):
    event_counts = event_counts.add(chunk["event_type"].value_counts(), fill_value=0)
    parts.append(chunk[chunk["user_id"] % KEEP_ONE_IN == 0])

events = pd.concat(parts, ignore_index=True).dropna(subset=["user_session"])
print("\nClick types in the WHOLE month:\n", event_counts.astype(int))
print("\nremove_from_cart exists:", "remove_from_cart" in event_counts.index)
print("Sampled clicks:", len(events), "| users:", events.user_id.nunique(), "| sessions:", events.user_session.nunique())

## Step 2 (runs from the repo): one row per abandoned visit, with the come-back label
Uses `build_visits` from `nudge/contract.py`, the rules decided after the data audit (20 Nov):
- a **visit** = a shopper's clicks with no gap over 60 minutes (the shop's own session id does not reset, so we ignore it);
- visits with a purchase are not scored (the shopper bought, no message would go out);
- **decision moment** = last click + 60 min (when Hostinger Reach would send its email);
- **label 1** = the same shopper comes back and buys within 7 days, with no message (the data has none);
- start word **returning** = a visit in the previous 14 days, else **new**;
- only visits with a full 14-day history and a full 7-day label window are kept.
Each click becomes one word: type (view/cart) + price band (cheap <€100, mid, expensive ≥€250) + `again` + time since previous click (start / quick <20 s / normal 20–70 s / long 70 s–30 min / away >30 min).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root, so we can import nudge
import pandas as pd
from nudge.contract import build_visits

events = pd.read_parquet("../data/sample/events_sample.parquet")
visits = build_visits(events)
print(visits.head())

## Step 3: the numbers that shape the whole project
- **Share that buy**: if it is small (e.g. 5%), a model that always says "won't buy" gets 95% accuracy and is useless, so we report precision, recall and F1.
- **Session length**: tells us how long the model's input must be.

In [ ]:
print("Abandoned visits:", len(visits))
print("Come back and buy within 7 days:", round(visits.label.mean(), 4))
cart = visits.cart_value > 0
print("Cart abandoners:", cart.sum(), "| of those come back and buy:", round(visits[cart].label.mean(), 4))
print(visits.groupby("split").label.agg(["size", "mean"]))
print(visits.tokens.explode().value_counts().head(10))

## Step 4: save (small files that go into the repo's `data/sample/`)

In [ ]:
visits.to_parquet("../data/sample/visits_sample.parquet", index=False)
print("saved", len(visits), "visits")